# Response Embedding Similarity Lookup

Given a response text that exists in `response_embeddings`, find the N most similar responses by cosine similarity.

In [1]:
import duckdb
import polars as pl

# DB_PATH = "f:/Project/games/jt3/data/jt3.duckdb"
# con = duckdb.connect(DB_PATH)

In [6]:
RESPONSE = "Abraham Lincoln"
N = 10

In [5]:
%load_ext sql

# Connect SQLMagic directly to the project DuckDB file.
%sql duckdb:///f:/Project/games/jt3/data/jt3.duckdb

%config SqlMagic.autopolars = True

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Connecting to 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

In [9]:
%%sql
WITH target AS (
    SELECT embedding
    FROM contextual_response_embeddings
    WHERE response_text = '{{RESPONSE}}'
)
SELECT
    r.response_text,
    array_cosine_similarity(r.embedding, t.embedding) AS similarity,
    1 - array_cosine_similarity(r.embedding, t.embedding) AS distance
FROM contextual_response_embeddings r, target t
WHERE r.response_text != '{{RESPONSE}}'
ORDER BY similarity DESC
LIMIT {{N}}

Running query in 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

response_text,similarity,distance
str,f32,f32
"""Lincoln""",0.932408,0.067592
"""Abe Lincoln""",0.815731,0.184269
"""Robert Todd Lincoln""",0.754012,0.245988
"""(Abraham) Lincoln""",0.744709,0.255291
"""""The Railsplitter""""",0.667729,0.332271
"""Buchanan""",0.667068,0.332932
"""Honest Abe""",0.666251,0.333749
"""Emancipator""",0.656129,0.343871
"""(James) Buchanan""",0.647934,0.352066


In [12]:
results = %sql WITH target AS ( \
                SELECT embedding \
                FROM contextual_response_embeddings \
                WHERE response_text = '{{RESPONSE}}' \
            ) \
            SELECT \
                r.response_text, \
                array_cosine_similarity(r.embedding, t.embedding) AS similarity, \
                1 - array_cosine_similarity(r.embedding, t.embedding) AS distance \
            FROM contextual_response_embeddings r, target t \
            WHERE r.response_text != '{{RESPONSE}}' \
            ORDER BY similarity DESC \
LIMIT {{N}}

results

Running query in 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

response_text,similarity,distance
str,f32,f32
"""Lincoln""",0.932408,0.067592
"""Abe Lincoln""",0.815731,0.184269
"""Robert Todd Lincoln""",0.754012,0.245988
"""(Abraham) Lincoln""",0.744709,0.255291
"""""The Railsplitter""""",0.667729,0.332271
"""Buchanan""",0.667068,0.332932
"""Honest Abe""",0.666251,0.333749
"""Emancipator""",0.656129,0.343871
"""(James) Buchanan""",0.647934,0.352066


### 3. Cross-Embedding Bridge

Use clue embeddings as a semantic bridge: find the target's clues → find similar clues → return those clues' answers. Surfaces responses related by question-topic rather than answer-text.

In [10]:
%%sql bridge_results <<
WITH target_clues AS (
    SELECT c.text AS clue_text
    FROM clues c
    WHERE c.correct_response = '{{RESPONSE}}'
),
target_clue_embeddings AS (
    SELECT ce.clue_text, ce.embedding
    FROM clue_embeddings ce
    WHERE ce.clue_text IN (SELECT clue_text FROM target_clues)
),
similar_clues AS (
    SELECT
        ce.clue_text,
        MAX(array_cosine_similarity(ce.embedding, tce.embedding)) AS similarity
    FROM clue_embeddings ce
    CROSS JOIN target_clue_embeddings tce
    WHERE ce.clue_text NOT IN (SELECT clue_text FROM target_clues)
    GROUP BY ce.clue_text
    ORDER BY similarity DESC
    LIMIT 100
)
SELECT
    c.correct_response AS response_text,
    MAX(sc.similarity) AS max_clue_similarity,
    AVG(sc.similarity) AS avg_clue_similarity,
    COUNT(*) AS bridge_clue_count
FROM similar_clues sc
JOIN clues c ON c.text = sc.clue_text
WHERE c.correct_response IS NOT NULL
    AND c.correct_response != '{{RESPONSE}}'
GROUP BY c.correct_response
ORDER BY max_clue_similarity DESC
LIMIT {{N}}

Running query in 'duckdb:///f:/Project/games/jt3/data/jt3.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Comparison

Side-by-side ranking of all three approaches.

In [13]:
contextual_ranked = (
    results
    .select("response_text", "similarity")
    .with_row_index("rank", offset=1)
    .rename({"response_text": "contextual", "similarity": "ctx_sim"})
)

prompted_ranked = (
    prompted_results
    .select("response_text", "prompted_similarity")
    .with_row_index("rank", offset=1)
    .rename({"response_text": "prompted", "prompted_similarity": "prm_sim"})
)

bridge_ranked = (
    bridge_results
    .select("response_text", "max_clue_similarity")
    .with_row_index("rank", offset=1)
    .rename({"response_text": "bridge", "max_clue_similarity": "brg_sim"})
)

comparison = (
    contextual_ranked
    .join(prompted_ranked, on="rank")
    .join(bridge_ranked, on="rank")
)

comparison

rank,contextual,ctx_sim,prompted,prm_sim,bridge,brg_sim
u32,str,f32,str,f32,str,f32
1,"""Lincoln""",0.932408,"""(Abraham) Lincoln""",0.991675,"""(Abraham) Lincoln""",0.954235
2,"""Abe Lincoln""",0.815731,"""Abe Lincoln""",0.967828,"""(Queen) Victoria""",0.935232
3,"""Robert Todd Lincoln""",0.754012,"""(Abe) Lincoln""",0.954215,"""Ohio""",0.934912
4,"""(Abraham) Lincoln""",0.744709,"""Lincoln""",0.946985,"""Hiram""",0.921731
5,"""""The Railsplitter""""",0.667729,"""Robert Lincoln""",0.924226,"""Illinois""",0.920892
6,"""Buchanan""",0.667068,"""Lincoln's Lincolns""",0.922848,"""Kentucky""",0.920183
7,"""Honest Abe""",0.666251,"""the Lincolns""",0.922215,"""B.""",0.917647
8,"""Emancipator""",0.656129,"""Andrew Lincoln""",0.918971,"""John Quincy Adams""",0.917178
9,"""(James) Buchanan""",0.647934,"""George Washington & Abraham Li…",0.88635,"""(Woodrow) Wilson""",0.915297


In [ ]:
%sql --close duckdb:///f:/Project/games/jt3/data/jt3.duckdb